In [29]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126 -q
!pip3 install scikit-image -q

In [39]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
import torch.nn as nn
import torch.nn.functional as F

from skimage.feature import hog
from skimage.color import rgb2gray

import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import confusion_matrix
import numpy as np
import random
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


torch.manual_seed(1)
random.seed(1)
np.random.seed(1)

def get_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Implementación CNN modular

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, num_conv_layers, first_layer_filters, kernel_size, norm):
        super(ConvBlock, self).__init__()

        layers = []
        in_ch = in_channels

        for i in range(num_conv_layers):
            out_ch = first_layer_filters * (2 ** i)

            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size, padding=1))

            if norm:
                layers.append(nn.BatchNorm2d(out_ch))

            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        self.block = nn.Sequential(*layers)
        out_ch = first_layer_filters * (2 ** (num_conv_layers - 1))
        self.last_filters = out_ch

    def forward(self, x):
        return self.block(x)


class FCBlock(nn.Module):
    def __init__(self, in_features, hidden_mlp_layers, output_neurons):
        super(FCBlock, self).__init__()

        layers = []
        for hidden_size in hidden_mlp_layers:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size

        layers.append(nn.Linear(in_features, output_neurons))
        # Sin Softmax (CrossEntropyLoss lo incluye internamente)

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [7]:
class ModuleCNN(nn.Module):
    def __init__(self, init_h, init_w, in_channels, num_conv_layers, first_layer_filters,
                 kernel_size, norm, hidden_mlp_layers, output_neurons):
        super(ModuleCNN, self).__init__()

        self.conv = ConvBlock(in_channels, num_conv_layers, first_layer_filters, kernel_size, norm)

        # Cálculo a mano del tamaño de la salida del bloque convolucional (alternativa al dummy)
        h, w = init_h, init_w
        for _ in range(num_conv_layers):
            # conv
            h = (h+2) - kernel_size + 1
            w = (w+2) - kernel_size + 1
            # pooling
            h = h //2
            w = w //2
        flat_size = self.conv.last_filters * h * w

        self.fc = FCBlock(flat_size, hidden_mlp_layers, output_neurons)

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [24]:
cnn = ModuleCNN(init_h=160, init_w=160, in_channels=3, num_conv_layers=5, first_layer_filters=8,
                kernel_size=3, norm=True, hidden_mlp_layers=[128], output_neurons=10)
print(f"O modelo ten {get_parameters(cnn):,} parámetros")

O modelo ten 509,898 parámetros


### DATASETS

In [44]:
# Descargar conjunto Imaginette
datasets.Imagenette(root="data/", split="train", size="160px", download=True)

# dividir en train y test
train_data = datasets.Imagenette(root="data/", split="train", size="160px", download=False, transform=transforms.Compose([transforms.Resize((160, 160)), transforms.ToTensor()]))
val_data = datasets.Imagenette(root="data/", split="val", size="160px", download=False, transform=transforms.Compose([transforms.Resize((160, 160)), transforms.ToTensor()]))

In [45]:
batch_size = 8
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=batch_size,
                              shuffle=True)
val_dataloader = DataLoader(dataset=val_data,
                             batch_size=batch_size,
                             shuffle=True)

### Ejecución del modelo